In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("🍳 스팀 게임 장르 예측 딥러닝 준비 완료! (정확도 영끌 버전🔥) 삐리릭-")


df = pd.read_csv('steam_games_preprocessed.csv')

target_col = "main_genre"
X = df.drop(columns=[target_col])
y = df[target_col]

# 결측치 0으로 채우고 float으로 변환
X = X.astype(float).fillna(0)

# 문자열 장르를 숫자로 변환
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

# 학습/테스트 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# 배치 사이즈를 128
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)


class SteamGenrePredictor(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SteamGenrePredictor, self).__init__()
        self.network = nn.Sequential(
            # 1층 (용량 2배 뻥튀기!)
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2), # 쉬는 시간 줄였긔!

            # 2층
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            # 3층 (새로 추가된 깊은 층!)
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # 출력층
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.network(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
input_dim = X_train.shape[1]
model = SteamGenrePredictor(input_dim, num_classes).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
print(f"🔥 공부 빡세게 시작! (총 {epochs} Epochs)")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"[Epoch {epoch+1:02d}/{epochs}] 평균 Loss: {running_loss/len(train_loader):.4f}")


# (정확도 평가)

model.eval()
correct = 0
total = 0

with torch.no_grad():
    inputs, labels = X_test_tensor.to(device), y_test_tensor.to(device)
    outputs = model(inputs)
    _, predicted = torch.max(outputs.data, 1)

    total = labels.size(0)
    correct = (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print("==================================================")
print(f"💖 스팀 게임 장르 예측 최종 정확도: {accuracy:.2f}% 💖")
print("==================================================")

🍳 스팀 게임 장르 예측 딥러닝 준비 완료! (정확도 영끌 버전🔥) 삐리릭-
🔥 공부 빡세게 시작! (총 50 Epochs)
[Epoch 01/50] 평균 Loss: 1.4817
[Epoch 10/50] 평균 Loss: 1.3722
[Epoch 20/50] 평균 Loss: 1.3470
[Epoch 30/50] 평균 Loss: 1.3270
[Epoch 40/50] 평균 Loss: 1.3091
[Epoch 50/50] 평균 Loss: 1.2923
💖 스팀 게임 장르 예측 최종 정확도: 45.58% 💖
